In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.2 Probability: Distributions, Expectation, and the Born Rule

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.2",
    title="Probability: Distributions, Expectation, and the Born Rule",
    blurb="Counts become probabilities. We build the distributions that physics "
    "runs on — binomial, Poisson, and the rest — and define expectation "
    "and variance in exactly the form quantum mechanics will use them: the "
    "expectation value and the uncertainty.",
    difficulty="intermediate",
    estimate="140–180 min",
)

## Notebook overview

[§5.1](counting.ipynb) taught us to count configurations. This one turns those counts into
**probabilities**, the language physics actually speaks, and it carries a payload worth
announcing up front. The two quantities we build here — the **expectation** $\langle A\rangle$
and the **variance**, whose square root is the standard deviation — are not merely the average
and the spread of a distribution. They are, letter for letter, the objects quantum mechanics
calls the *expectation value of an observable* and its *uncertainty* $\Delta A$. The
Heisenberg uncertainty principle is a statement about variances. So we define these things in
their physics form from the first line, and when observables become operators in Volume VI,
the definitions will transfer without a single change.

We build patiently. First the bridge from counting: for equally likely outcomes a probability
is just favourable microstates over total microstates, and the requirement that all
probabilities sum to one is — we flag it immediately — the very normalization $\langle\psi|
\psi\rangle=1$ that the Born rule will demand of a quantum state. Then conditional probability
and **Bayes' theorem**, treated carefully as the disciplined updating of a sample space (not
as the apparatus of statistical inference, which belongs to a different subject). Then the
distributions physical systems actually realize: the **binomial** of a two-state paramagnet,
the **multinomial** of an energy partition, the **geometric** waiting time, and — in full —
the **Poisson** law of rare events that governs radioactive decay, photon counting, and shot
noise. We show the binomial *becomes* Poisson in the rare-event limit, and we close on the
expectation and variance themselves, the Born-rule heart of the notebook.

Throughout we lean on **Monte Carlo** — estimating probabilities by simulation with
`numpy.random.default_rng` — both as confirmation and as a preview, since Monte Carlo is the
engine that drives this volume's physics from [§5.4](microstates-entropy-temperature.ipynb) onward. We verify our hand-built
distributions against `scipy.stats` (used only as an independent check, never as a
substitute for building them ourselves), and we reuse the `ecp.combinatorics` schematics and
distribution-bar plots from [§5.1](counting.ipynb).

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: probabilities summing to one; $P(\text{sum}=8)=5/36$; Bayes reversing the
> conditioning; the hand-built binomial and Poisson matching `scipy.stats`; the Poisson
> signature $\langle k\rangle=\mathrm{Var}=\lambda$; the die's $\langle A\rangle=3.5$ and
> $\mathrm{Var}=35/12$; variances of independent variables adding. A ✓ is strong evidence; a
> ✗ is a prompt to *locate the discrepancy*, not a verdict.
>
> **Scope.** Probability as the language of microstate statistics; the large-$N$ limit
> (Stirling, the CLT, the sharpness of macrostates) is [§5.3](large-n-limit.ipynb), and the physics begins at [§5.4](microstates-entropy-temperature.ipynb).
> See Feller, *An Introduction to Probability Theory*; Schroeder, *Thermal Physics*; and
> Volume VI (the Born rule, where $\langle A\rangle$ and $\Delta A$ return as operators).

## Theory in brief

### From counts to probability

For equally likely outcomes, a probability is a ratio of counts,

```{math}
:label: eq-probability
P(\text{event})=\frac{\text{favourable microstates}}{\text{total microstates}}, \qquad \sum_i p_i = 1 .
```

A sample space lists the mutually exclusive outcomes; their probabilities are non-negative
and **sum to one**. Flag this normalization now: it is exactly the quantum normalization
$\langle\psi|\psi\rangle=1$. Probabilities are the structure the Born rule will make physical.

### Conditional probability and Bayes' theorem

Conditioning on an event $B$ restricts the sample space to $B$,

```{math}
:label: eq-conditional
P(A\mid B)=\frac{P(A\cap B)}{P(B)}, \qquad P(A\mid B)=\frac{P(B\mid A)\,P(A)}{P(B)}, \qquad P(A\cap B)=P(A)P(B)\ \text{(independent)} .
```

**Bayes' theorem** (the middle equation) reverses the conditioning; here it is pure
conditional probability — the updating of a sample space — not statistical inference.
Independence, the last equation, is the physical statement that separate subsystems multiply
their probabilities, which becomes product states and additive entropy.

### The physical distributions

Four distributions carry most of statistical physics,

```{math}
:label: eq-distributions
\text{binomial } \binom{n}{k}p^k(1-p)^{n-k}, \quad \text{geometric } (1-p)^{k-1}p, \quad \text{Poisson } \frac{\lambda^k e^{-\lambda}}{k!} .
```

The **binomial** counts successes in $n$ independent two-outcome trials (a two-state system),
with $\langle k\rangle=np$ and $\mathrm{Var}=np(1-p)$. The **multinomial** generalises it to
several outcomes (an energy partition). The **geometric** is the waiting time to the first
success, mean $1/p$. The **Poisson** is the limit of the binomial for many rare events
($n\to\infty$, $p\to0$, $np=\lambda$ fixed); its signature is $\langle k\rangle=\mathrm{Var}=
\lambda$, and it governs radioactive decay, photon arrivals, and shot noise.

### Expectation and variance — the Born-rule heart

The two summary numbers are

```{math}
:label: eq-expectation
\langle A\rangle=\sum_i a_i\,P(a_i), \qquad \mathrm{Var}(A)=\langle A^2\rangle-\langle A\rangle^2=(\Delta A)^2 .
```

Read them as physics. $\langle A\rangle$ is the **expectation value** of an observable, and
$\Delta A=\sqrt{\mathrm{Var}(A)}$ is its **uncertainty** — the very $\Delta x$, $\Delta p$ of
the Heisenberg relation. Expectation is linear, and for *independent* variables variances add,
$\mathrm{Var}(X+Y)=\mathrm{Var}(X)+\mathrm{Var}(Y)$.

### Monte Carlo: probability by simulation

A probability or expectation can be estimated by sampling,

```{math}
:label: eq-monte-carlo
\langle A\rangle \approx \frac{1}{N}\sum_{i=1}^{N} A(\omega_i), \qquad \text{error} \sim \frac{1}{\sqrt N} ,
```

the estimate converging to the truth as $1/\sqrt N$ — the seed of the Monte Carlo methods
that become this volume's spine from [§5.4](microstates-entropy-temperature.ipynb).

## Setup

In [ ]:
from math import comb, exp, factorial

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy import stats

from ecp import combinatorics as cb
from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = cb.RED


def binomial_pmf(k, n, p):
    """The binomial probability P(k) = C(n,k) p^k (1-p)^(n-k) (eq-distributions).

    The probability of exactly ``k`` successes in ``n`` independent trials each succeeding
    with probability ``p`` — the distribution of a two-state system (a paramagnet's spins).

    Parameters
    ----------
    k : int or numpy.ndarray
        Number of successes.
    n : int
        Number of trials.
    p : float
        Single-trial success probability.

    Returns
    -------
    float or numpy.ndarray
        The probability P(k).
    """
    k = np.asarray(k)
    coeff = np.array([comb(n, int(ki)) for ki in np.atleast_1d(k)]).reshape(k.shape)
    return coeff * p**k * (1.0 - p) ** (n - k)


def poisson_pmf(k, lam):
    """The Poisson probability P(k) = λ^k e^(-λ)/k! (eq-distributions).

    The probability of ``k`` events when they occur independently at mean number ``lam`` —
    the law of rare events (decay counts, photon arrivals, shot noise).

    Parameters
    ----------
    k : int or numpy.ndarray
        Number of events.
    lam : float
        The mean number of events λ.

    Returns
    -------
    float or numpy.ndarray
        The probability P(k).
    """
    k = np.asarray(k)
    fact = np.array([factorial(int(ki)) for ki in np.atleast_1d(k)]).reshape(k.shape)
    return lam**k * exp(-lam) / fact


def expectation(values, probs):
    """The expectation value ⟨A⟩ = Σ_i a_i P(a_i) (eq-expectation).

    The Born-rule expectation of an observable taking ``values`` with probabilities ``probs``.

    Parameters
    ----------
    values : array_like
        The possible values a_i of the observable.
    probs : array_like
        Their probabilities P(a_i) (summing to one).

    Returns
    -------
    float
        The expectation ⟨A⟩.
    """
    return float(np.sum(np.asarray(values) * np.asarray(probs)))


def variance(values, probs):
    """The variance Var(A) = ⟨A^2⟩ − ⟨A⟩^2 = (ΔA)^2 (eq-expectation).

    The square of the uncertainty ΔA of the observable — the quantity the Heisenberg
    relation bounds.

    Parameters
    ----------
    values : array_like
        The possible values of the observable.
    probs : array_like
        Their probabilities.

    Returns
    -------
    float
        The variance Var(A).
    """
    values, probs = np.asarray(values), np.asarray(probs)
    mean = expectation(values, probs)
    return float(np.sum(values**2 * probs) - mean**2)

## Exercise 1 — From counts to probability

Probability begins exactly where [§5.1](counting.ipynb) left off. When outcomes are equally likely, the
probability of an event is the fraction of microstates that realise it {eq}`eq-probability`,
so every count we computed last notebook is one division away from a probability. Take poker:
the probability of being dealt a flush is its count over the total, $5108/2598960\approx
0.0020$. Because every hand falls into exactly one category, the category probabilities must
**sum to one** ({numref}`fig-pr-pokerprob`) — and that requirement is not a bookkeeping nicety.
It is the same condition the Born rule will impose on a quantum state, $\langle\psi|\psi\rangle
=1$: the probabilities of all the outcomes of a measurement must add to certainty.

**This exercise (worked).** Convert the poker counts of [§5.1](counting.ipynb) to probabilities with `math.comb`,
tabulate them, confirm the high-card remainder reproduces the classic count $1{,}302{,}540$
(the genuine test that the eight named-hand counts are consistent with the total), and
confirm the probabilities sum to $1$. Flag the parallel: this normalization is the classical
shadow of quantum normalization.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    counts["high card"] == 1302540 and abs(sum(probs.values()) - 1.0) < 1e-12,
    "the categories complete the deck (high card = 1,302,540) and the probabilities sum to 1",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Conditional probability

A probability can change the moment we learn something. **Conditional probability** $P(A\mid
B)$ is the probability of $A$ once we know $B$ has happened, computed by restricting the sample
space to $B$ {eq}`eq-conditional`. Roll two dice: across all $36$ equally likely outcomes, the
sum is $8$ in five of them ($2{+}6,3{+}5,4{+}4,5{+}3,6{+}2$), so $P(\text{sum}=8)=5/36$. But if
we are *told* the first die shows $5$, the sample space shrinks to six outcomes, and only one
of them ($5{+}3$) sums to $8$, so $P(\text{sum}=8\mid\text{first}=5)=1/6$
({numref}`fig-pr-dice`). Learning the first die changed the probability.

**This exercise (worked).** Enumerate the $36$ outcomes with `itertools.product`, compute
$P(\text{sum}=8)$ and the conditional $P(\text{sum}=8\mid\text{first}=5)$ by counting within
the restricted space, and confirm **independence** of the two dice by checking $P(\text{both
even})=P(\text{even})^2$. *Forward-pointer:* independent subsystems multiplying their
probabilities is what makes entropy additive ([§5.4](microstates-entropy-temperature.ipynb)).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(p_sum8, 5 / 36, "P(sum=8) = 5/36", rtol=1e-9)
validate.close(
    p_both_even,
    0.25,
    "the two dice are independent: P(both even) = P(even)² = 1/4",
    rtol=1e-9,
)

## Exercise 3 — Bayes' theorem

This one rewards going slowly. **Bayes' theorem** reverses a conditional probability: it turns
"the probability of the evidence given the cause" into "the probability of the cause given the
evidence" {eq}`eq-conditional`. Here is a clean version. Two bags sit on a table. Bag $A$ holds
two red balls and one blue; bag $B$ holds one red and two blue ({numref}`fig-pr-bags`). We pick
a bag at random — each with probability $\tfrac12$ — and draw one ball, which turns out red.
What is the probability we drew from bag $A$?

Work it in pieces. The probability of red *given* $A$ is $P(R\mid A)=2/3$, and given $B$ it is
$P(R\mid B)=1/3$. The overall probability of drawing red is the weighted average $P(R)=P(R\mid
A)P(A)+P(R\mid B)P(B)=\tfrac23\cdot\tfrac12+\tfrac13\cdot\tfrac12=\tfrac12$. Bayes then assembles
these into the answer, $P(A\mid R)=P(R\mid A)P(A)/P(R)=(\tfrac23\cdot\tfrac12)/\tfrac12=\tfrac23$.
The red draw made bag $A$ twice as likely as bag $B$, exactly because $A$ is twice as rich in
red. This is conditional probability — the disciplined updating of a sample space once evidence
arrives — not the separate machinery of statistical inference.

**This exercise (worked).** Compute $P(A\mid R)$ by Bayes' theorem as explicit Python
arithmetic, and confirm it against a **direct enumeration** — iterate over every equally likely
(bag, ball) outcome and take the fraction of red draws that came from $A$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    p_A_given_red,
    enum_A_given_red,
    "Bayes' theorem reverses the conditioning correctly (matches direct enumeration)",
    rtol=1e-12,
)
validate.close(
    p_A_given_red,
    2 / 3,
    "the red draw makes bag A twice as likely: P(A|red) = 2/3",
    rtol=1e-9,
)

## Exercise 4 — The binomial distribution

Our first full distribution, and the workhorse of statistical physics. Flip a coin $n$ times,
each flip independently heads with probability $p$; the probability of getting exactly $k$
heads is the **binomial** $\binom{n}{k}p^k(1-p)^{n-k}$ {eq}`eq-distributions` — the $C(n,k)$
sequences of [§5.1](counting.ipynb), each weighted by its probability. Its mean is $\langle k\rangle=np$ and its
variance $\mathrm{Var}=np(1-p)$. This is exactly a **two-state system**: $n$ spins each up or
down, with $k$ up. The distribution we plot here is, with spins for coins, the multiplicity
distribution of a paramagnet ({numref}`fig-pr-binom`).

**This exercise (worked).** For $n=20$, $p=0.4$, build the binomial pmf with the `binomial_pmf`
helper, verify it against `scipy.stats.binom.pmf` (an independent check), and confirm the mean
and variance. The animation flips coins with `numpy.random.default_rng` and watches the
histogram of head-counts build up toward the binomial curve.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    pmf_binom,
    pmf_scipy,
    "the hand-built binomial pmf matches the standard scipy.stats one",
    rtol=1e-9,
)
validate.close(
    np.array([mean_b, var_b]),
    np.array([n_b * p_b, n_b * p_b * (1 - p_b)]),
    "the binomial has ⟨k⟩=np and Var=np(1−p)",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The multinomial and geometric distributions

Two quick generalisations. The **multinomial** distribution counts the ways $n$ trials split
among *several* outcomes rather than two — the natural object when energy is partitioned among
many levels. Roll six dice; the probability of seeing exactly one of each face is
$6!\,(1/6)^6\approx0.0154$: the $6!$ orderings of "one of each" out of the $6^6$ equally likely
rolls. The **geometric** distribution is the waiting time to the first success in repeated
trials, $P(k)=(1-p)^{k-1}p$, with mean $1/p$ — wait, on average, $1/p$ trials for an event of
probability $p$.

**This exercise (worked).** Compute the multinomial probability of one-of-each on six dice with
`math.factorial`, verify it against a brute-force enumeration of all $6^6=46{,}656$ rolls with
`itertools.product`, and compute the geometric mean $\sum_k k(1-p)^{k-1}p=1/p$ as an explicit
`numpy` sum, confirming it equals $1/p$. *Forward-pointer:* the multinomial is the
combinatorial form of partitioning energy among many levels ([§5.4](microstates-entropy-temperature.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    np.array([p_one_each, geometric_mean]),
    np.array([enum_one_each, 1 / p_geo]),
    "the multinomial matches brute-force enumeration and the geometric mean equals 1/p",
    rtol=1e-6,
)

## Exercise 6 — The Poisson distribution

The **Poisson** distribution is the law of rare events, and it deserves a full treatment
because physics is full of it. When events happen independently at some average rate, the
number $k$ observed in a fixed interval follows $P(k)=\lambda^k e^{-\lambda}/k!$
{eq}`eq-distributions`, where $\lambda$ is the mean count. Its signature, which no other
distribution shares, is that the **mean equals the variance**, $\langle k\rangle=\mathrm{Var}=
\lambda$ ({numref}`fig-pr-poisson`). A detector clicking on average $\lambda=3$ times a second,
a sample emitting $\lambda$ decays per minute, photons arriving at a sensor — all Poisson, and
all forward pointers to the quantum counting of Volumes VI and VII.

**This exercise (worked).** For $\lambda=3$ (say, detector clicks per second) build the Poisson
pmf with the `poisson_pmf` helper, verify it against `scipy.stats.poisson.pmf`, confirm the
signature $\langle k\rangle=\mathrm{Var}=\lambda$, and cross-check with a Monte Carlo using
`rng.poisson`.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    pmf_poisson,
    stats.poisson.pmf(k_p, lam),
    "the hand-built Poisson pmf matches the standard scipy.stats one",
    rtol=1e-9,
)
validate.close(
    np.array([poisson_mean, poisson_var]),
    np.array([lam, lam]),
    "the Poisson distribution has mean = variance = λ (its signature)",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The binomial → Poisson limit

Now the conceptual heart, worth taking slowly. Poisson is not an unrelated distribution; it is
what the binomial *becomes* in the limit of **many rare trials**. Imagine a great many
opportunities for an event, each individually very unlikely, with the average number of events
held fixed: $n\to\infty$, $p\to0$, $np=\lambda$ constant. A radioactive sample has $\sim10^{23}$
nuclei (huge $n$), each with a tiny chance of decaying in the next second (tiny $p$), and a
definite average decay rate ($\lambda=np$). In that limit the binomial $\binom{n}{k}p^k(1-p)^
{n-k}$ converges term by term to $\lambda^k e^{-\lambda}/k!$ {eq}`eq-distributions`
({numref}`fig-pr-limit`); Feller, *An Introduction to Probability Theory*, Vol. I, Ch. VI,
carries the term-by-term limit out in full. This is *why* decay counts, photon arrivals, and shot noise are all
Poisson: each is a many-rare-events process.

**This exercise (worked).** Fix $\lambda=3$ and evaluate the binomial pmf (the `binomial_pmf`
helper) for growing $n$ ($n=10,30,1000$ with $p=\lambda/n$), comparing each to Poisson$(3)$
with `numpy.max` of the absolute difference. Confirm that by $n=1000$ the two agree to better
than $10^{-3}$ everywhere, and watch the binomial bars settle onto the Poisson curve.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    binom_1000,
    poisson_ref,
    "the binomial converges to the Poisson distribution in the rare-event limit",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Expectation and variance: the Born-rule constructs

Here is the payload of the notebook, and it is worth stating plainly. The **expectation**
$\langle A\rangle=\sum_i a_i P(a_i)$ and the **variance** $\mathrm{Var}(A)=\langle A^2\rangle-
\langle A\rangle^2$ {eq}`eq-expectation` are not just the average and spread of a distribution.
They are, exactly and without modification, the quantities quantum mechanics calls the
**expectation value** of an observable and the square of its **uncertainty** $\Delta A$. When
Volume VI promotes observables to operators and writes $\langle A\rangle=\langle\psi|\hat A|
\psi\rangle$, the *meaning* is this same weighted average; and the Heisenberg uncertainty
relation $\Delta x\,\Delta p\ge\hbar/2$ is a statement about these very variances. We are not
learning a classical preliminary to be discarded — we are learning the definitions QM uses,
in a setting simple enough to see them whole.

**This exercise (worked).** Compute $\langle A\rangle$ and $\mathrm{Var}(A)$ for a fair die
(the `expectation` and `variance` helpers), confirming $\langle A\rangle=3.5$ and
$\mathrm{Var}=35/12\approx2.917$, and report $\Delta A=\sqrt{\mathrm{Var}}$; then do the same
for the binomial of Exercise 4. *Frame (voice):* the expectation value and the uncertainty are
not quantum inventions. They are these elementary definitions, met again unchanged when
observables become operators.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array([E_die, Var_die]),
    np.array([3.5, 35 / 12]),
    "⟨A⟩=Σ aP(a) and Var=⟨A²⟩−⟨A⟩² — the Born-rule expectation (3.5) and the uncertainty (35/12)",
    rtol=1e-9,
)

## Exercise 9 — Variance of sums and the seed of sharpness

We end the conceptual arc with the fact that makes thermodynamics possible. For **independent**
variables, variances add: $\mathrm{Var}(X+Y)=\mathrm{Var}(X)+\mathrm{Var}(Y)$ {eq}`eq-expectation`.
So if we average $N$ independent, identically distributed measurements, the *sum* has variance
$N\sigma^2$ (it grows), but the *mean* — the sum over $N$ — has variance $\sigma^2/N$ (it
shrinks). The relative fluctuation of the mean therefore falls as $1/\sqrt N$.
For $N\sim10^{23}$ that is a relative spread of $\sim10^{-12}$:
the average becomes, for all practical purposes, a sharp number. This is the seed of why
macrostates are sharp and thermodynamics is deterministic — the theme [§5.3](large-n-limit.ipynb) develops in full.

**This exercise (student).** With `numpy.random.default_rng`, draw many samples of $N$ dice and
confirm that the variance of the *sum* is $N$ times the single-die variance, while the relative
standard deviation of the *mean* falls as $1/\sqrt N$ across $N=1,10,100,1000$.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    var_of_sum_1000 / 1000,
    var_single,
    "variances of independent variables add: Var(sum) = N·Var(single)",
    rtol=5e-2,
)
validate.close(
    rel_fluct[3] / rel_fluct[1],
    1 / np.sqrt(100),
    "the relative fluctuation of the mean falls as 1/√N",
    rtol=1.5e-1,
)

## Exercise 10 — Monte Carlo estimation in practice

Finally, a glance ahead at the method that will carry this whole volume. Everything above we
could compute exactly, but most physical systems are too complicated for that, and then we
**estimate** probabilities and expectations by sampling {eq}`eq-monte-carlo`. The estimate is
itself a random quantity, and by the variance-of-the-mean result of Exercise 9 its error
shrinks as $1/\sqrt N$ — slow, but utterly general, and indifferent to the dimensionality that
defeats other methods. Monte Carlo turns probability into a computational instrument, and from
[§5.4](microstates-entropy-temperature.ipynb) it becomes the engine of statistical-mechanics simulation.

**This exercise (student).** Estimate $\pi$ by the classic dart-throwing Monte Carlo — the
fraction of random points in the unit square that land inside the quarter circle is $\pi/4$ —
with `numpy.random.default_rng`, and confirm the error falls roughly as $1/\sqrt N$ across
decades of sample size.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.close(
    4 * ((rng.random((2_000_000, 2)) ** 2).sum(axis=1) <= 1.0).mean(),
    np.pi,
    "the Monte Carlo estimate of π converges to the exact value",
    rtol=1e-2,
)

## Notebook summary

Probability turns the counts of [§5.1](counting.ipynb) into the language physics speaks, and it hands quantum
mechanics two of its central definitions ready-made.

- **From counts to probability** {eq}`eq-probability`: equally likely outcomes give $P=$
  favourable/total; all probabilities **sum to one** — the classical ancestor of $\langle\psi|
  \psi\rangle=1$.
- **Conditional probability and Bayes** {eq}`eq-conditional`: conditioning restricts the sample
  space ($P(\text{sum}=8\mid\text{first}=5)=1/6$); Bayes reverses it ($P(A\mid\text{red})=2/3$),
  as disciplined updating, not inference; independence multiplies ($P(\text{both even})=1/4$).
- **The physical distributions** {eq}`eq-distributions`: the **binomial** (two-state system,
  $\langle k\rangle=np$, verified against `scipy.stats`), the multinomial and geometric, and
  the **Poisson** law of rare events ($\langle k\rangle=\mathrm{Var}=\lambda$), which the
  binomial *becomes* as $n\to\infty,p\to0,np=\lambda$ (agreement to $10^{-3}$ at $n=1000$).
- **Expectation and variance — the Born-rule heart** {eq}`eq-expectation`: $\langle A\rangle=
  \sum a_iP(a_i)$ and $\mathrm{Var}=\langle A^2\rangle-\langle A\rangle^2$ ($3.5$ and $35/12$
  for a die) are the QM expectation value and the uncertainty $\Delta A$ — Heisenberg is a
  variance statement. For independent variables variances add, so the mean of $N$ sharpens as
  $1/\sqrt N$, the seed of why macrostates are sharp.
- **Monte Carlo** {eq}`eq-monte-carlo`: probability as a computational tool, converging as
  $1/\sqrt N$ — the engine of the physics from [§5.4](microstates-entropy-temperature.ipynb).

The distributions are the ones physical systems realize; the expectation and uncertainty,
defined here in physics form, will be carried unchanged into Volume VI; and Monte Carlo is the
instrument of everything that follows. What remains before the physics is the **large-$N$
limit** — why, when $N\sim10^{23}$, the most probable configuration does not merely win but
utterly dominates. That is [§5.3](large-n-limit.ipynb).

## Outlook

- **The large-$N$ limit ([§5.3](large-n-limit.ipynb)).** Stirling's approximation, the law of large numbers, the
  central limit theorem, and why a macrostate's overwhelming multiplicity makes thermodynamics
  deterministic.
- **The physics begins ([§5.4](microstates-entropy-temperature.ipynb) onward).** Microstate probabilities become the Boltzmann distribution and
  entropy; Monte Carlo becomes the simulation method for the Ising model and beyond.
- **The Born rule (Volume VI).** Probabilities as $|\text{amplitude}|^2$, $\langle A\rangle$ as
  $\langle\psi|\hat A|\psi\rangle$, and the uncertainty principle as a variance relation — the
  constructs of this notebook made quantum.
- **Poisson processes (Volumes VI–VII).** Photon counting and quantum optics, where the Poisson
  law of this notebook is the statistics of coherent light.
- **Cross-reference** [§5.1](counting.ipynb) (counting, the source of these probabilities) and Volume VI (the Born
  rule).

In [ ]:
from ecp.style import footer

footer()